# 04 · Ujednolicenie i integracja danych akustycznych

<div style="border-left: 4px solid #2b6cb0; padding: 12px 16px; margin-bottom: 20px; border-radius: 0 4px 4px 0;">
  <strong>Praca Inżynierska:</strong> Analiza i klasyfikacja dźwięków z wykorzystaniem uczenia maszynowego <br>
  <strong>Autorzy:</strong> Norbert Gwiazda, Mateusz Ciurzyński | <strong>Data:</strong> 2026-11-04<br>
  <strong>Zbiory danych:</strong> Respiratory Sound Database (ICBHI 2017), HF_Lung_V2, SPRSound<br>
  <strong>Cel:</strong> Analiza eksploracyjna zbiorów danych
</div>

---

## Spis treści
- [1. Wprowadzenie](#1-wprowadzenie)
---


##  1. Wprowadzenie

W notatnikach od 01 do 03 przedstawiono różne zbiory danych zawierające nagrania cykli oddechowych (ICBHI 2017, HF-Lung-V2 oraz SPRSound). Każdy z tych zasobów charakteryzuje się odmienną specyfiką techniczną — począwszy od różnic w częstotliwości próbkowania i czasie trwania nagrań, aż po zróżnicowanie rejestrowanych dźwięków płucnych, gdzie wybrane patologie (np. furczenia) występują wyłącznie w części baz.

Niniejszy notatnik opisuje proces integrowania poszczególnych źródeł w jeden spójny zbiór danych. Wszystkie podjęte decyzje inżynieryjne oraz założenia projektowe zostały w nim szczegółowo omówione i uzasadnione.

---

## 2. Środowisko i Import Bibliotek

In [19]:
%matplotlib inline
%config InlineBackend.figure_format = 'svg'
%config InlineBackend.close_figures = True


import os
import sys
import warnings
from pathlib import Path
from datetime import datetime
from itertools import chain

import librosa
import matplotlib.gridspec as gridspec

# Wizualizacja danych
import matplotlib.pyplot as plt

# Przetwarzanie i analiza numeryczna
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import scipy.stats as stats
import seaborn as sns
import statsmodels.api as sm
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from pyfonts import load_google_font
from pypalettes import load_cmap, load_palette
import json


# Skrypty lokalne
root_dir = Path().resolve().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.dataset import SPRSoundDataset, ICBHIDataset, HFLungSoundDataset

# Wyciszenie zbędnych ostrzeżeń
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

# Wczytanie czionek i palet kolorów
dark_two = load_palette("Dark2")
regular = load_google_font("Roboto", weight=400)
medium = load_google_font("Roboto", weight=500)
bold = load_google_font("Roboto", weight=700)
heavy = load_google_font("Roboto", weight=800)

# Ustawienie motywu
sns.set_theme(style="whitegrid")

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42


## 3. Wczytanie danych


In [22]:
CACHE_DIR = '.cache'

os.makedirs(CACHE_DIR, exist_ok=True)

BASE_DIR = Path('../data')

ICBHI_PATH = BASE_DIR / "ICBHI_final_database"
DIAGNOSIS_FILE = ICBHI_PATH / 'ICBHI_Challenge_diagnosis.txt'

HF_LUNG_DATA_1 = BASE_DIR / "HF_Lung_V1-master"
HF_LUNG_DATA_2 = BASE_DIR / "HF_Lung_V1_IP-main"

TRAIN_PATH_1 = HF_LUNG_DATA_1 / "train"
TEST_PATH_1 = HF_LUNG_DATA_1 / "test"

TRAIN_PATH_2 = HF_LUNG_DATA_2 / "train"
TEST_PATH_2 = HF_LUNG_DATA_2 / "test"

SPR_PATH = BASE_DIR / "SPRSound"

audio_files = list(ICBHI_PATH.rglob("*.wav"))

SPRSoundDataset = SPRSoundDataset(SPR_PATH)
ICBHIDataset = ICBHIDataset(ICBHI_PATH, DIAGNOSIS_FILE)
HFLungSoundDataset1 = HFLungSoundDataset(BASE_DIR)

df1 = ICBHIDataset.get_df()
df2 = HFLungSoundDataset1.get_df()
df3 = SPRSoundDataset.get_df()

TypeError: 'SPRSoundDataset' object is not callable

## 4. Scalanie zbiorów

### 4.1 Różnice w kolumnach

In [21]:
print(df1.columns)
print(df2.columns)
print(df3.columns)

Index(['start', 'end', 'crackles', 'wheezes', 'filename', 'patient_id',
       'recording_id', 'chest_location', 'acquisition_mode',
       'recording_equipment', 'wav_path', 'diagnosis'],
      dtype='str')
Index(['start', 'end', 'filename', 'wav_path', 'recording_equipment', 'date',
       'truncation', 'chest_location', 'split', 'wheezes', 'crackles'],
      dtype='str')
Index(['start', 'end', 'filename', 'wav_path', 'patient_id', 'patient_age',
       'patient_gender', 'chest_location', 'wheezes', 'crackles'],
      dtype='str')


Proces unifikacji heterogenicznych zbiorów danych (ICBHI, HF Lung i SPRSound) rozpoczyna się od standaryzacji struktur indeksujących, takich jak `filename`, `start` oraz `end`. Te kolumny pełnią funkcję fundamentalnego wektora przestrzenno-czasowego, który umożliwia precyzyjne wycięcie pojedynczych cykli oddechowych i zachowanie pełnej spójności segmentacji audio niezależnie od pierwotnego formatu źródłowego.

Kluczowym krokiem inżynieryjnym jest rygorystyczne oddzielenie zmiennych objaśniających od etykiet docelowych, ponieważ kolumny opisujące dźwięki patologiczne (`crackles`, `wheezes`, `rhonchi`, `stridor`) nie stanowią wejścia do modelu, lecz tworzą przestrzeń klasyfikacyjną (multi-label targets). Wymaga to przekształcenia osobnych flagnikowych cech binarnych z ICBHI i HF Lung oraz zbiorczego `event_type` ze SPRSound w jeden, spójny wektor docelowy dla całego potoku uczenia maszynowego.

Pozostałe kolumny metadanych, w tym `date`, `diagnosis`, `patient_age`, `truncation`, `acquisition_mode` oraz `patient_gender`, zostają słusznie odrzucone z powodów metodologicznych. Usunięcie diagnozy zapobiega groźnemu wyciekowi danych podczas rzeczywistej inferencji klinicznej, natomiast pominięcie demografii i daty eliminuje problem brakujących wartości w rozproszonych źródłach, pozwalając architekturze sieci opierać się wyłącznie na czystej ekstrakcji cech akustycznych ze spektrogramów.